In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/full_dedup")

validation_pairs = pd.read_parquet(
    DATA_DIR / "validation_pairs.parquet"
)

deduplicated = pd.read_parquet(
    DATA_DIR / "deduplicated_records.parquet"
)

print(validation_pairs.shape)
print(deduplicated.shape)

(15000, 3)
(3653581, 18)


In [2]:
left = deduplicated.add_prefix("left_")
right = deduplicated.add_prefix("right_")

review_df = (
    validation_pairs
    .merge(
        left,
        left_on="idx1",
        right_index=True,
        how="left"
    )
    .merge(
        right,
        left_on="idx2",
        right_index=True,
        how="left"
    )
)

cols = [
    "label",

    "left_country",
    "right_country",

    "left_party_name",
    "right_party_name",

    "left_name_latin",
    "right_name_latin",

    "left_cluster_id",
    "right_cluster_id",
]

review_df = review_df[cols]

review_df.head(20)

,label,left_country,right_country,left_party_name,right_party_name,left_name_latin,right_name_latin,left_cluster_id,right_cluster_id
0,0,arm,arm,Աղասի Մքոյան,Աղասի Գրիգորյան,aghasi mqoyan,aghasi grigoryan,5506,18796
1,0,arm,arm,Դավիթ Ջալալյան,«ՔԱՖԵՎԻ»,davit jalalyan,qafevi,16981,17397
2,0,arm,arm,Ալեքսանդր Նաումով,Ալեքսանդր Տրունով,aleqsandr naoumov,aleqsandr trounov,3291,173861
3,1,kgz,kgz,Бабаев Расулбек Махмудович,Бабаев Расулбек Махмудович,babaev rasulbek mahmudovich,babaev rasulbek mahmudovich,1172920,1172920
4,0,mng,kaz,дагий цогтбаяр,АНТРОПОВ АНДРЕЙ НИКОЛАЕВИЧ,dagii tsogtbayar,antropov andrei nikolaevich,1459645,436273
5,0,kaz,kaz,ДУСКАЛИЕВА АЙГУЛЬ РАМАЗАНОВНА,АМАНТАЕВА АНЕЛЯ АЙБЕКҚЫЗЫ,duskalieva ai gul ramazanovna,amantaeva anelya ai bekkyzy,474438,376912
6,1,mng,mng,Нацагдорж Соёлмаа,нацагдорж соёлмаа,natsagdorzh soe lmaa,natsagdorzh soe lmaa,1301758,1301758
7,1,kgz,kgz,Ли Юань,Ли Юн ---,li yuan,li yun,1000659,1000659
8,1,arm,arm,Նոննա Քոչարյան,Նոննա Քոչարյան,nonna qocharyan,nonna qocharyan,7933,7933
9,1,kgz,kgz,Дуйшоев Орозали Эркинович,Дуйшоев Орозали Эркинович,dui shoev orozali erkinovich,dui shoev orozali erkinovich,1127742,1127742


In [5]:
manual_review = pd.concat([
    # positives
    review_df[review_df["label"] == 1].sample(
        150,
        random_state=42,
    ),

    # negatives
    review_df[review_df["label"] == 0].sample(
        150,
        random_state=42,
    ),
])

manual_review = manual_review.sample(
    frac=1,
    random_state=42,
).reset_index(drop=True)

manual_review["manual_label"] = ""

print(manual_review.shape)

manual_review

(300, 10)


,label,left_country,right_country,left_party_name,right_party_name,left_name_latin,right_name_latin,left_cluster_id,right_cluster_id,manual_label
0,0,arm,arm,Ալբերտ Նալբանդյան,Ալբերտ Մեջլումյան,albert nalbandyan,albert mejloumyan,21741,334,
1,0,arm,arm,Ադել Մուհամեդ Ալի Ջասիմ Ալ Մարզուգի,Ադել Մոհամեդ Ալի Ջասիմ Ալմարզուքի,adel mouhamed ali jasim al marzougi,adel mohamed ali jasim almarzouqi,14805,8164,
2,0,arm,arm,Աիդա Միրզոյան,ԱԻԴԱ ՀԱՄԲԱՐՁՈՒՄՅԱՆ,aida mirzoyan,aida hambardzoumyan,7686,3048,
3,1,mng,mng,Лхамсүрэн Уянга,лхамсүрэн уянга,lhamsuren uyanga,lhamsuren uyanga,1299374,1299374,
4,0,arm,arm,«ԱԼՎԱՆԴ ՄԱՅՆԻՆԳ ԵՎ ՄԻՆԵՐԱԼ ԻՆԴԱՍՏՐԻԶ»,Ալվարդ Սարգսյան,alvand mayning ev mineral indastriz,alvard sargsyan,6489,30582,
...,...,...,...,...,...,...,...,...,...,...
295,0,arm,arm,Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),a t s nominis limitid kipros,a t s nominis limitid kipros,129520,181194,
296,1,mng,mng,мандах мөнхбат,мандах мөнхбат,mandah monhbat,mandah monhbat,1274136,1274136,
297,1,mng,mng,Ганболд Баттулга,ганболд баттулга,ganbold battulga,ganbold battulga,1288685,1288685,
298,0,kaz,mng,ДИРР ЮРИЙ ВЛАДИМИРОВИЧ,баточир баттогтох,dirr yurii vladimirovich,batochir battogtoh,599888,1525045,


In [4]:
OUTPUT_PATH = (
    DATA_DIR / "manual_gold_pairs.csv"
)

manual_review.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_PATH)

Saved: ..\data\full_dedup\manual_gold_pairs.csv
